# 04. FGSM 적대적 공격 & 견고성 평가
- FGSM(Fast Gradient Sign Method) 구현
- epsilon 값별 공격 실험 (0.01 ~ 0.1)
- Robustness Index 계산 (제안서 4.3 반영)
- 공격 전후 Grad-CAM 비교 시각화

In [ ]:
# ============================================================
# 공통 모듈 import
# ============================================================
import sys
sys.path.append("..")

import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

from src.dataset import get_dataloaders
from src.model   import load_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"디바이스: {device}")

In [ ]:
# ============================================================
# DataLoader & 모델 로드
# ============================================================
train_loader, val_loader, test_loader, train_ds, val_ds, test_ds = get_dataloaders(
    shard_root  = "../data/train",
    shard_nums  = [0],
    batch_size  = 32,
    num_workers = 0
)

model     = load_model("../results/best_resnet50.pth", device=device)
criterion = nn.CrossEntropyLoss()

In [ ]:
# ============================================================
# FGSM 구현
# x_adv = x + epsilon * sign(∇_x L(x, y))
# ============================================================
def fgsm_attack(model, imgs: torch.Tensor, labels: torch.Tensor, epsilon: float) -> torch.Tensor:
    """
    FGSM: 손실 함수의 그래디언트 부호 방향으로 epsilon만큼 노이즈 추가

    Args:
        model  : 학습된 모델
        imgs   : 입력 이미지 텐서 (B, 3, 224, 224)
        labels : 정답 라벨 (B,)
        epsilon: 노이즈 강도 (클수록 공격 강도 증가)
    """
    imgs = imgs.clone().requires_grad_(True)   # 그래디언트 추적 활성화
    outputs = model(imgs)
    loss    = criterion(outputs, labels)
    model.zero_grad()
    loss.backward()                            # 입력 이미지에 대한 그래디언트 계산
    perturbed = imgs + epsilon * imgs.grad.sign()  # 부호 방향으로 노이즈 추가
    perturbed = torch.clamp(perturbed, -3.0, 3.0)  # 정규화 범위 클리핑
    return perturbed.detach()


def compute_robustness_index(clean_auc: float, attack_auc: float) -> float:
    """RI = 공격 후 AUC / 공격 전 AUC (1에 가까울수록 견고)"""
    return attack_auc / clean_auc if clean_auc > 0 else 0.0

In [ ]:
# ============================================================
# Clean 성능 측정 (공격 전 기준)
# ============================================================
clean_probs, clean_labels_list = [], []

with torch.no_grad():
    for imgs, labels in tqdm(test_loader, desc="Clean 평가"):
        imgs, labels = imgs.to(device), labels.to(device)
        outputs = model(imgs)
        probs   = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()
        clean_probs.extend(probs)
        clean_labels_list.extend(labels.cpu().numpy())

clean_auc = roc_auc_score(clean_labels_list, clean_probs)
print(f"공격 전 Clean AUC: {clean_auc:.4f}")

In [ ]:
# ============================================================
# epsilon 별 FGSM 공격 실험
# ============================================================
epsilons = [0.01, 0.03, 0.05, 0.1]
results  = {}

for eps in epsilons:
    attack_probs, attack_labels_list = [], []

    for imgs, labels in tqdm(test_loader, desc=f"FGSM ε={eps}"):
        imgs, labels = imgs.to(device), labels.to(device)
        adv_imgs     = fgsm_attack(model, imgs, labels, epsilon=eps)

        with torch.no_grad():
            outputs = model(adv_imgs)
            probs   = torch.softmax(outputs, dim=1)[:, 1].cpu().numpy()

        attack_probs.extend(probs)
        attack_labels_list.extend(labels.cpu().numpy())

    attack_auc = roc_auc_score(attack_labels_list, attack_probs)
    ri         = compute_robustness_index(clean_auc, attack_auc)
    results[eps] = {"attack_auc": attack_auc, "robustness_index": ri}
    print(f"ε={eps:.2f} | Attack AUC: {attack_auc:.4f} | Robustness Index: {ri:.4f}")

In [ ]:
# ============================================================
# 결과 시각화
# ============================================================
eps_list = list(results.keys())
auc_list = [results[e]["attack_auc"]       for e in eps_list]
ri_list  = [results[e]["robustness_index"]  for e in eps_list]

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(eps_list, auc_list, marker="o", color="red",    label="Attack AUC")
axes[0].axhline(y=clean_auc, color="blue", linestyle="--",   label=f"Clean AUC ({clean_auc:.3f})")
axes[0].set_title("FGSM 공격 강도별 ROC AUC")
axes[0].set_xlabel("Epsilon")
axes[0].set_ylabel("ROC AUC")
axes[0].set_ylim(0, 1)
axes[0].legend()

axes[1].plot(eps_list, ri_list,  marker="s", color="orange", label="Robustness Index")
axes[1].axhline(y=1.0, color="green", linestyle="--",        label="RI=1.0 (완벽한 견고성)")
axes[1].set_title("Robustness Index")
axes[1].set_xlabel("Epsilon")
axes[1].set_ylabel("RI")
axes[1].set_ylim(0, 1.1)
axes[1].legend()

plt.tight_layout()
plt.savefig("../results/robustness_analysis.png", dpi=150)
plt.show()

In [ ]:
# ============================================================
# 공격 전후 Grad-CAM 비교 시각화
# ============================================================
IMAGENET_MEAN = np.array([0.485, 0.456, 0.406])
IMAGENET_STD  = np.array([0.229, 0.224, 0.225])

target_layers = [model.layer4[-1]]
cam = GradCAM(model=model, target_layers=target_layers)

def denorm(tensor):
    """정규화된 텐서를 시각화 가능한 numpy 배열로 변환"""
    x = tensor.squeeze().permute(1, 2, 0).cpu().numpy()
    return (x * IMAGENET_STD + IMAGENET_MEAN).clip(0, 1).astype(np.float32)

# 테스트셋 첫 배치에서 3장 비교
sample_imgs, sample_labels = next(iter(test_loader))
sample_imgs   = sample_imgs.to(device)
sample_labels = sample_labels.to(device)
n = 3

adv_imgs = fgsm_attack(model, sample_imgs[:n], sample_labels[:n], epsilon=0.05)

for i in range(n):
    orig = sample_imgs[i:i+1]
    adv  = adv_imgs[i:i+1]

    cam_orig = cam(input_tensor=orig, targets=[ClassifierOutputTarget(1)])[0]
    cam_adv  = cam(input_tensor=adv,  targets=[ClassifierOutputTarget(1)])[0]

    with torch.no_grad():
        p_orig = torch.softmax(model(orig), dim=1)[0, 1].item()
        p_adv  = torch.softmax(model(adv),  dim=1)[0, 1].item()

    orig_np = denorm(orig)
    adv_np  = denorm(adv)

    fig, axes = plt.subplots(1, 4, figsize=(18, 4))
    fig.suptitle(f"정답: {'Real' if sample_labels[i]==0 else 'AI'} | ε=0.05", fontsize=12)

    axes[0].imshow(orig_np)
    axes[0].set_title(f"원본 (AI확률: {p_orig:.3f})")
    axes[0].axis("off")
    axes[1].imshow(show_cam_on_image(orig_np, cam_orig, use_rgb=True))
    axes[1].set_title("원본 Grad-CAM")
    axes[1].axis("off")
    axes[2].imshow(adv_np)
    axes[2].set_title(f"공격 후 (AI확률: {p_adv:.3f})")
    axes[2].axis("off")
    axes[3].imshow(show_cam_on_image(adv_np, cam_adv, use_rgb=True))
    axes[3].set_title("공격 후 Grad-CAM")
    axes[3].axis("off")

    plt.tight_layout()
    plt.savefig(f"../results/attack_gradcam_sample{i}.png", dpi=150)
    plt.show()